# Tips 11：複数PDFを結合

**業務の困りごと**：配布資料のPDFを1冊にまとめる。

**このTipsでできること**：`PdfWriter.append` で番号順にページを追加し、1つのPDFへ書き込みます。

**使用ライブラリ**：pypdf, reportlab

### ファイルの場所
`/content/` はColabの一時領域です。PCやGoogle Driveには自動保存されません。左側のフォルダから結果を選んでダウンロードできます。セッション終了前に保存してください。

上から順番に実行してください。各Tipsのサンプルはノートブック内で作ります。


## 1. 練習用データと準備

1ページずつの練習用PDFを3つ作ります。ファイル名の先頭の数字が結合順です。

最後の `display()` で練習用データ全体を表として表示します。列名・行数・値を確認してから次へ進みましょう。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
%pip -q install pypdf reportlab
from pathlib import Path
from reportlab.pdfgen import canvas
from pypdf import PdfReader,PdfWriter
base=Path('/content') if Path('/content').exists() else Path.cwd()
# 元PDFを置くフォルダ
folder=base/'tips11_PDF'; folder.mkdir(exist_ok=True)
# 3ページ分の練習用PDFを作る
for i in range(1,4):
    path=folder/f'{i:02d}_page.pdf'
    c=canvas.Canvas(str(path)); c.drawString(100,740,f'Sample document {i}'); c.save()
import pandas as pd
print('結合前のPDF一覧（先頭5件）:')
display(pd.DataFrame({'ファイル名':[p.name for p in sorted(folder.glob('*_page.pdf'))]}))


## 2. 自動処理

`PdfWriter.append` で番号順にページを追加し、1つのPDFへ書き込みます。

**結果を見るポイント**：結合後のPDFが **3ページ** で、順番が01→02→03か確認しましょう。

保存した成果物を読み直して `display()` で表示します。画面の表とダウンロードしたファイルの内容を照らし合わせてください。


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print  # Colab以外の環境向け
# 結合先のPDFを準備する
writer=PdfWriter()
# ファイル名の順番どおりにページを追加する
for path in sorted(folder.glob('*_page.pdf')):
    writer.append(str(path))
out=base/'Tips11_結合資料.pdf'
# 完成PDFを書き出す
with out.open('wb') as f: writer.write(f)
print('ページ数:',len(PdfReader(out).pages))

# 完成したPDFの各ページから、文字情報を確認する
display(pd.DataFrame([{'ページ番号': i + 1, '文字の例': page.extract_text()}
                      for i, page in enumerate(PdfReader(out).pages)]))


## 3. 結果の確認

`assert` は条件が正しいか確認する命令です。条件が違えばそこで止まるため、想定外の集計を見つけられます。

結合後のPDFが **3ページ** で、順番が01→02→03か確認しましょう。


In [ ]:
# 件数や合計が期待値と一致するか検査する
assert len(PdfReader(out).pages)==3
print('確認OK:',out)


## 4. 持ち帰り・練習

**練習問題**：ページ順を逆にして結合しましょう。



**実務で使う前に**：順番が重要な業務では、ファイル名にゼロ埋め番号を付けると並びが安定します。

出力をPCへ保存したい場合は次のセルをColabで実行します。Colab以外では出力場所を表示します。


In [ ]:
# Colab上で作った成果物をPCへダウンロードする
try:
    from google.colab import files
    files.download(str(out))
except ImportError:
    print('出力ファイル:',out.resolve())
